# 🔬 Advanced Research Tools

This notebook covers research-grade tooling for paper reproducibility and deeper analysis — not part of the everyday Discover → Evaluate → Intervene loop.

- **Section A:** Selector Registry — all 14 selectors, custom registration
- **Section B:** MasterGrid — (methods × wrappers × seeds) experiment grid
- **Section C:** Intervention Faithfulness (IF) — LOO cross-validated metric
- **Section D:** Pipeline reuse — `from_artifact` / `from_scores` iteration

- **Model:** GPT-2 (CPU-friendly where possible)
- **Runtime:** ~10 min (Sections B/C use small grids or synthetic data)

---

## Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import circuitkit as ck
from circuitkit import Pipeline

---
## Section A: Selector Registry

CircuitKit has 14 registered selectors — functions that rank nodes by different criteria (magnitude, random, algorithm-specific, etc.). They're used internally by Pillar 5 (baselines) and can be used directly for custom experiments.

In [ ]:
from circuitkit.selection import list_selectors, get_selector

# List all registered selectors
all_selectors = list_selectors()
print(f"{len(all_selectors)} selectors registered:")
for name in sorted(all_selectors):
    print(f"  - {name}")

In [ ]:
# Get a specific selector function
magnitude_selector = get_selector("magnitude")
random_selector = get_selector("random")

print(f"magnitude: {magnitude_selector}")
print(f"random:    {random_selector}")

### Registering a Custom Selector

You can register your own selector with `@register("name")`. A selector is a callable with signature `(model, task, config_dict) -> Dict[str, float]`.

In [ ]:
from circuitkit.selection import register

@register("my_custom")
def my_custom_selector(model, task, config):
    """Example selector: (model, task, config_dict) -> Dict[str, float].

    In practice, use `model` and `task` to compute real importance scores
    (e.g. activation norms, gradients). This toy version returns uniform
    placeholder scores.
    """
    n_layers = getattr(getattr(model, "cfg", None), "n_layers", 10)
    return {f"node_{i}": 1.0 / (i + 1) for i in range(n_layers)}

# Verify registration
print(f"Now {len(list_selectors())} selectors registered")
assert "my_custom" in list_selectors()

---
## Section B: MasterGrid

The `MasterGrid` runs a (methods × wrappers × seeds) experiment grid. It's designed for systematic comparison across the full CircuitKit toolkit.

A full grid on a real model is expensive, so here we demonstrate the API with a small configuration.

In [ ]:
from circuitkit.evaluation.master_grid import MasterGrid, MasterGridCell

In [ ]:
# Small demo grid — 2 methods × 2 wrappers × 2 seeds
grid = MasterGrid(
    methods=["eap", "eap-ig"],
    model="gpt2",
    wrappers=["pruning", "quantization"],
    seeds=[42, 123],
)

In [ ]:
# Run the grid (small scale — takes ~2-3 min on CPU)
import os
os.makedirs("./results/master_grid", exist_ok=True)

try:
    grid.run(out_dir="./results/master_grid")
    print(f"Grid completed: {len(grid.cells)} cells")
except Exception as e:
    print(f"Grid run failed (expected on limited hardware): {e}")
    print("Proceeding with synthetic data for demo...")

    # Populate with synthetic results for demonstration
    import random
    grid.cells = []
    for m in ["eap", "eap-ig"]:
        for w in ["pruning", "quantization"]:
            for s in [42, 123]:
                grid.cells.append(MasterGridCell(
                    method=m, wrapper=w, seed=s,
                    quality=random.uniform(0.5, 0.9),
                    stability=random.uniform(0.3, 0.8),
                    faithfulness=random.uniform(0.4, 0.85),
                ))

In [ ]:
# View as DataFrame
df = grid.to_dataframe()
df

In [ ]:
# Per-wrapper ranking summary
print(grid.summary())

In [ ]:
# Per-wrapper ranks dict
ranks = grid.per_wrapper_ranks()
for (method, wrapper), rank in sorted(ranks.items()):
    print(f"  {method:>8s} × {wrapper:>14s} → rank {rank}")

### Loading Pre-Computed Results

For large grids, results are typically computed via shell scripts and aggregated later.

`MasterGrid.from_csv("path/to/aggregated_grid.csv")` loads a CSV with columns: `method`, `wrapper`, `seed`, `quality`, `stability`, ...

---
## Section C: Intervention Faithfulness (IF) Metric

The IF metric predicts whether a circuit will produce good downstream interventions, using three cheap signals:

$$IF(c, w) = \alpha \cdot J_5(c) + \beta \cdot R_3(c) + \gamma \cdot P(c, w)$$

where $J_5$ is Pillar-5 stability, $R_3$ is Pillar-3 baseline ratio, and $P$ is a low-budget intervention probe.

Coefficients are fit by leave-one-wrapper-out cross-validation.

In [ ]:
from circuitkit.evaluation.intervention_faithfulness import IF, IFResult

In [ ]:
# Construct a synthetic cells dict for demonstration
# Format: {(method, wrapper): {"stability": float, "baseline_ratio": float,
#           "probe": float, "quality": float, "faithfulness": float}}
import random
random.seed(42)

methods = ["eap", "eap-ig", "eap-gp"]
wrappers = ["pruning", "quantization", "lora", "steering"]

cells = {}
for m in methods:
    for w in wrappers:
        cells[(m, w)] = {
            "stability": random.uniform(0.3, 0.9),
            "baseline_ratio": random.uniform(0.5, 1.5),
            "probe": random.uniform(0.2, 0.8),
            "quality": random.uniform(0.4, 0.9),
            "faithfulness": random.uniform(0.4, 0.9),
        }

print(f"Cells: {len(cells)} entries ({len(methods)} methods × {len(wrappers)} wrappers)")

In [ ]:
# Fit the IF metric with leave-one-wrapper-out CV
if_metric = IF(simplex_grid_step=0.1)
result = if_metric.fit_loo(cells)

print(f"IF accuracy:          {result.if_accuracy:.1%}")
print(f"Faith-only accuracy:  {result.faith_accuracy:.1%}")
print(f"Stab-only accuracy:   {result.stab_accuracy:.1%}")
print(f"Chance (top-1):       {result.chance_top1:.1%}")

In [ ]:
# Per-fold details — compare IF prediction against faithfulness-only and stability-only baselines
header = f"{'Held wrapper':<14} {'Coeffs (a,b,g)':<20} {'True':>10} {'IF':>10} {'Faith':>10} {'Stab':>10}"
print(header)
print("-" * len(header))
for fold in result.folds:
    coeffs_str = f"({fold.coeffs[0]:.2f},{fold.coeffs[1]:.2f},{fold.coeffs[2]:.2f})"
    print(f"{fold.held_wrapper:<14} {coeffs_str:<20} {fold.true_winner:>10} "
          f"{fold.pred_if:>10} {fold.pred_faithfulness_only:>10} {fold.pred_stability_only:>10}")

---
## Section D: Pipeline Reuse — Iterate Without Re-Discovering

Once you've discovered a circuit, you can reload it and try different interventions without running discovery again.

In [ ]:
# First, create an artifact to work with
pipe_orig = Pipeline("gpt2", task="ioi", precision="float32", output_dir="./results/reuse")
pipe_orig.discover(algorithm="eap-ig", n_examples=32, batch_size=4, ig_steps=5)

artifact_path = pipe_orig._artifact_path
print(f"Artifact saved at: {artifact_path}")

### `Pipeline.from_artifact()` — Reload and Apply

In [ ]:
# Reload the circuit and prune at different sparsity
pipe_v2 = Pipeline.from_artifact(
    artifact_path,
    model_name="gpt2",
    task="ioi",
    precision="float32",
    output_dir="./results/reuse/v2",
)

print(f"Loaded circuit: {pipe_v2._circuit}")

# Apply a different sparsity without re-discovering
pipe_v2.prune(sparsity=0.5, scope="both")
pipe_v2.export("./results/reuse/v2/checkpoint_sparse50")

pipe_v2.summary()

### `Pipeline.from_scores()` — Load from Scores File

In [ ]:
# The scores side-car (_scores.pt) can also be loaded
from pathlib import Path

scores_pt = Path(artifact_path).parent / f"{Path(artifact_path).stem}_scores.pt"
if scores_pt.exists():
    pipe_v3 = Pipeline.from_scores(
        scores_pt,
        model_name="gpt2",
        task="ioi",
        precision="float32",
    )
    print(f"Loaded from scores: {pipe_v3._circuit}")
else:
    print(f"Scores file not found at {scores_pt}")
    print("This happens when discovery doesn't produce a _scores.pt side-car.")

---
## Summary

| Tool | Purpose | When to use |
|------|---------|-------------|
| Selector Registry | Access / register node-ranking functions | Custom baselines, new algorithms |
| MasterGrid | Systematic (method × wrapper × seed) comparison | Paper experiments |
| IF Metric | Predict intervention quality from cheap signals | Method selection |
| `from_artifact` / `from_scores` | Reload without re-discovering | Iteration, sparsity sweeps |